# Demo — RAG mínimo con SQLite y MiniLM

**Clase 3** — Arquitectura de Aplicaciones con IA Generativa

**Objetivo del demo:** implementar un RAG funcional desde cero, sin librerías mágicas, para *ver* cada pieza.

**Stack:**
- `sentence-transformers` con `paraphrase-multilingual-MiniLM-L12-v2` (384 dim, multilingüe, ~420 MB)
- `sqlite3` (stdlib) como vector store — embeddings guardados como BLOB
- `numpy` para la similitud coseno a mano
- DeepSeek vía OpenAI-compatible para la generación final

**Lo que NO usamos** (a propósito):
- LangChain, LlamaIndex, Haystack — queremos ver el código, no ocultarlo.
- `sqlite-vec`, `pgvector`, Qdrant — mencionados al final como siguiente paso.
- BM25, reranking — simplificamos a retrieval puramente vectorial.

<style>
.jp-RenderedMarkdown { font-size: 1.1em; line-height: 1.5; }
.jp-RenderedMarkdown h1 { font-size: 2.1em; }
.jp-RenderedMarkdown h2 { font-size: 1.6em; }
.jp-RenderedMarkdown h3 { font-size: 1.25em; }
</style>

## §0 — Setup

Cargamos credenciales y verificamos que tenemos la API key.

In [1]:
import os
import sqlite3
import numpy as np
from dotenv import load_dotenv
from rich import print

load_dotenv()

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
assert DEEPSEEK_API_KEY, "Falta DEEPSEEK_API_KEY en .env"
print(f"API key cargada: {DEEPSEEK_API_KEY[:4]}...{DEEPSEEK_API_KEY[-4:]}")

API key cargada: sk-a...7f4c

## §1 — Corpus de ejemplo

Usamos un corpus pequeño sobre el propio curso, así podemos verificar a ojo si el retrieval funciona.

En producción el corpus vendría de PDFs, Confluence, BD, etc. El patrón es el mismo.

In [2]:
CORPUS = """\
La Clase 1 del curso Arquitectura de Aplicaciones con IA Generativa introduce los fundamentos de los LLM: cómo funcionan por dentro, qué es un token, y por qué los proveedores convergen en la API estilo OpenAI. Se revisan los principales modelos disponibles: Claude, GPT, Gemini, DeepSeek y modelos open-source como Llama.

La Clase 2 se centra en Prompt Engineering como sistema. Enseña a tratar los prompts como código: versionados, testeables, compuestos en pipelines. Se cubren zero-shot, few-shot, chain-of-thought y self-consistency. También se trabaja el output estructurado con JSON mode y Pydantic + Instructor para validación con retry.

La Clase 3 aborda patrones de arquitectura con LLMs: RAG (retrieval augmented generation), orquestación de herramientas y gestión de memoria. El bloque central es la implementación de un RAG mínimo con SQLite y sentence-transformers, mostrando explícitamente cómo se embeben los chunks, se guardan como BLOB y se recuperan por similitud coseno.

La Clase 4 trata la construcción de aplicaciones LLM completas: backend, frontend, streaming Server-Sent Events (SSE), persistencia de conversaciones en base de datos, gestión de estado y rate limiting. Se toma el RAG de la Clase 3 y se convierte en un producto con interfaz.

La Clase 5 cierra el curso con despliegue y producción: observabilidad (logging, tracing, métricas), evaluación automática de calidad de respuestas, control de costes, latencia, y estrategias de fallback entre proveedores cuando una API está caída.

El stack técnico recomendado durante el curso incluye Python para prototipos y notebooks, DeepSeek como proveedor principal por su relación precio-calidad, SQLite para persistencia ligera, y FastAPI para los backends de la Clase 4.
"""

print(f"Corpus: {len(CORPUS)} caracteres, {len(CORPUS.split())} palabras")

Corpus: 1753 caracteres, 262 palabras

### §1.1 — Chunking

Partimos el texto en fragmentos pequeños. Estrategia mínima: por párrafos, con una longitud objetivo en caracteres y overlap opcional.

**Por qué chunks y no el documento entero:**
- El embedding de un texto largo captura su significado *promedio*, perdiendo detalles.
- Granularidad fina → el retrieval puede traer solo la parte relevante.
- Respeta la ventana de contexto del LLM en la fase de generación.

**Tradeoffs del tamaño de chunk:**
- Muy pequeño (50 tokens) → pierde contexto, retrieval ruidoso.
- Muy grande (2000 tokens) → embedding difuso, desperdicia contexto en generación.
- *Sweet spot* típico: 200–800 tokens.

In [3]:
def chunk_text(text: str, max_chars: int = 400, overlap: int = 50) -> list[str]:
    """Chunking ingenuo por párrafos, con merge hasta max_chars y overlap en caracteres."""
    paragraphs = [p.strip() for p in text.split("\n\n") if p.strip()]
    chunks: list[str] = []
    buffer = ""
    for p in paragraphs:
        if len(buffer) + len(p) + 2 <= max_chars:
            buffer = (buffer + "\n\n" + p).strip()
        else:
            if buffer:
                chunks.append(buffer)
            buffer = p
    if buffer:
        chunks.append(buffer)

    if overlap > 0 and len(chunks) > 1:
        with_overlap = [chunks[0]]
        for i in range(1, len(chunks)):
            tail = chunks[i - 1][-overlap:]
            with_overlap.append(tail + " " + chunks[i])
        chunks = with_overlap
    return chunks


chunks = chunk_text(CORPUS, max_chars=400, overlap=50)
print(f"Número de chunks: {len(chunks)}")
for i, c in enumerate(chunks):
    print(f"\n[chunk {i}] ({len(c)} chars)\n{c[:150]}...")

Número de chunks: 6

(322 chars)
La Clase 1 del curso Arquitectura de Aplicaciones con IA Generativa introduce los fundamentos de los LLM: cómo 
funcionan por dentro, qué es un token, ...

(373 chars)
Gemini, DeepSeek y modelos open-source como Llama. La Clase 2 se centra en Prompt Engineering como sistema. Enseña 
a tratar los prompts como código: v...

(395 chars)
y Pydantic + Instructor para validación con retry. La Clase 3 aborda patrones de arquitectura con LLMs: RAG 
(retrieval augmented generation), orquesta...

(326 chars)
dan como BLOB y se recuperan por similitud coseno. La Clase 4 trata la construcción de aplicaciones LLM completas: 
backend, frontend, streaming Server...

(299 chars)
lase 3 y se convierte en un producto con interfaz. La Clase 5 cierra el curso con despliegue y producción: 
observabilidad (logging, tracing, métricas)...

(282 chars)
lback entre proveedores cuando una API está caída. El stack técnico recomendado durante el curso incluye Python 
para prototipos y notebooks, DeepSeek ...

## §2 — Embeddings

Un **embedding** es un vector que representa el significado de un texto. Lo generamos con un modelo entrenado específicamente para esto — aquí `paraphrase-multilingual-MiniLM-L12-v2`, que:

- Produce vectores de **384 dimensiones**.
- Está entrenado en **50+ idiomas** (incluido español).
- Es **local**: corre en CPU, sin API externa, sin coste por llamada.
- Pesa **~420 MB**: primera ejecución descarga a `~/.cache/huggingface/`.

In [4]:
from sentence_transformers import SentenceTransformer

MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
embedder = SentenceTransformer(MODEL_NAME)
print(f"Modelo cargado: {MODEL_NAME}")
print(f"Dimensión: {embedder.get_sentence_embedding_dimension()}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Modelo cargado: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2

/var/folders/12/0n5dnc_d59zfr24jcj4fz22w0000gp/T/ipykernel_11758/3161574801.py:6: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Dimensión: {embedder.get_sentence_embedding_dimension()}")


Dimensión: 384

### Ver un embedding

Embebemos una frase y mostramos el vector resultante — para romper la abstracción de "embedding" y ver que es, literalmente, una lista de números.

In [5]:
ejemplo = "La Clase 3 trata sobre RAG."
vec = embedder.encode(ejemplo, normalize_embeddings=True)

print(f"Frase: {ejemplo!r}")
print(f"Shape: {vec.shape}")
print(f"Dtype: {vec.dtype}")
print(f"Primeros 10 valores: {vec[:10]}")
print(f"Norma (debe ser ~1): {np.linalg.norm(vec):.6f}")

Frase: 'La Clase 3 trata sobre RAG.'

Shape: (384,)

Dtype: float32

Primeros 10 valores: [-0.0218821   0.00114369 -0.01915164 -0.05482449  0.01251379 -0.01839098
 -0.03989692 -0.02273492 -0.02984717  0.03536474]

Norma (debe ser ~1): 1.000000

### Embeber el corpus

Ahora embebemos **todos** los chunks de una vez. `encode()` acepta una lista y procesa por batch.

In [6]:
embeddings = embedder.encode(chunks, normalize_embeddings=True, show_progress_bar=False)
print(f"Embeddings shape: {embeddings.shape}")
print(f"Cada chunk → vector de {embeddings.shape[1]} dimensiones")

Embeddings shape: (6, 384)

Cada chunk → vector de 384 dimensiones

## §3 — Vector store en SQLite

No usamos una base de datos vectorial especializada. Guardamos los embeddings como **BLOB** en una tabla SQLite normal.

**Schema:**
- `id` — PK autoincrement
- `text` — el chunk original (para inyectar en el prompt)
- `embedding` — el vector serializado como bytes (`np.float32` → `tobytes()`)

Esto es pedagógicamente claro, pero **no escala**. En producción querrás `sqlite-vec`, `pgvector` o una DB vectorial dedicada — lo vemos al final.

In [7]:
DB_PATH = "rag.db"

conn = sqlite3.connect(DB_PATH)
cur = conn.cursor()

cur.execute("DROP TABLE IF EXISTS chunks")
cur.execute("""
    CREATE TABLE chunks (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        text TEXT NOT NULL,
        embedding BLOB NOT NULL
    )
""")

for text, vec in zip(chunks, embeddings):
    blob = vec.astype(np.float32).tobytes()
    cur.execute("INSERT INTO chunks (text, embedding) VALUES (?, ?)", (text, blob))

conn.commit()
print(f"Insertados {len(chunks)} chunks en {DB_PATH}")

Insertados 6 chunks en rag.db

In [8]:
for row in cur.execute("SELECT id, length(embedding) AS bytes, substr(text, 1, 60) AS preview FROM chunks"):
    print(row)

(1, 1536, 'La Clase 1 del curso Arquitectura de Aplicaciones con IA Gen')

(2, 1536, 'Gemini, DeepSeek y modelos open-source como Llama. La Clase ')

(3, 1536, 'y Pydantic + Instructor para validación con retry. La Clase ')

(4, 1536, 'dan como BLOB y se recuperan por similitud coseno. La Clase ')

(5, 1536, 'lase 3 y se convierte en un producto con interfaz. La Clase ')

(6, 1536, 'lback entre proveedores cuando una API está caída. El stack ')

## §4 — Retrieval con similitud coseno

Dada una pregunta del usuario:

1. La embebemos con el **mismo modelo** (crítico: modelos distintos producen vectores en espacios distintos).
2. Calculamos la similitud coseno contra **todos** los chunks.
3. Devolvemos los **top-K** más similares.

Como los embeddings están normalizados, el coseno = producto punto.

In [9]:
def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))


def retrieve(query: str, k: int = 3) -> list[tuple[float, str]]:
    q_vec = embedder.encode(query, normalize_embeddings=True)

    rows = cur.execute("SELECT text, embedding FROM chunks").fetchall()
    scored: list[tuple[float, str]] = []
    for text, blob in rows:
        vec = np.frombuffer(blob, dtype=np.float32)
        score = cosine_similarity(q_vec, vec)
        scored.append((score, text))

    scored.sort(key=lambda x: x[0], reverse=True)
    return scored[:k]

In [10]:
query = "¿Qué se enseña sobre despliegue y observabilidad?"
resultados = retrieve(query, k=3)

print(f"Query: {query}\n")
for i, (score, text) in enumerate(resultados):
    print(f"[{i}] score={score:.4f}")
    print(f"    {text[:200]}...\n")

Query: ¿Qué se enseña sobre despliegue y observabilidad?

[0] score=0.4475

lase 3 y se convierte en un producto con interfaz. La Clase 5 cierra el curso con despliegue y producción: 
observabilidad (logging, tracing, métricas), evaluación automática de calidad de respuestas, ...

[1] score=0.3701

Gemini, DeepSeek y modelos open-source como Llama. La Clase 2 se centra en Prompt Engineering como sistema. 
Enseña a tratar los prompts como código: versionados, testeables, compuestos en pipelines. S...

[2] score=0.3364

y Pydantic + Instructor para validación con retry. La Clase 3 aborda patrones de arquitectura con LLMs: RAG 
(retrieval augmented generation), orquestación de herramientas y gestión de memoria. El bloq...

## §5 — Generación con vs sin RAG

Ahora comparamos: misma pregunta, mismo modelo, única diferencia → si inyectamos o no los chunks recuperados en el prompt.

Este contraste es el **momento pedagógico clave** de la clase.

In [12]:
from openai import OpenAI

client = OpenAI(api_key=DEEPSEEK_API_KEY, base_url="https://api.deepseek.com")
MODEL = "deepseek-chat"


def answer_without_rag(query: str) -> str:
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": "Responde de forma concisa en español."},
            {"role": "user", "content": query},
        ],
        temperature=0,
    )
    return response.choices[0].message.content


def answer_with_rag(query: str, k: int = 3) -> str:
    retrieved = retrieve(query, k=k)
    context = "\n\n---\n\n".join(text for _, text in retrieved)
    system = (
        "Responde la pregunta del usuario usando ÚNICAMENTE la información del CONTEXTO. "
        "Si el contexto no contiene la respuesta, di que no lo sabes. Responde en español y sé conciso.\n\n"
        f"CONTEXTO:\n{context}"
    )
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": query},
        ],
        temperature=0,
    )
    return response.choices[0].message.content

In [13]:
pregunta = "¿En qué clase se aprende sobre streaming SSE y persistencia de conversaciones?"

print("=== SIN RAG ===")
print(answer_without_rag(pregunta))
print("\n=== CON RAG ===")
print(answer_with_rag(pregunta))

=== SIN RAG ===

En la clase de **Desarrollo de Aplicaciones Web en Tiempo Real** o **Arquitectura de Sistemas Web Avanzados** se 
suele aprender sobre **SSE (Server-Sent Events)** y **persistencia de conversaciones**. También puede abordarse en 
cursos de **Backend con Node.js** o **APIs RESTful**, donde se integran tecnologías como Redis o bases de datos 
para mantener el estado de las conversaciones.

=== CON RAG ===

Según el contexto, se aprende sobre streaming SSE y persistencia de conversaciones en la **Clase 4**.

In [14]:
pregunta_2 = "¿Qué se enseña sobre patrones de orquestación y memoria?"

print("=== SIN RAG ===")
print(answer_without_rag(pregunta_2))
print("\n=== CON RAG ===")
print(answer_with_rag(pregunta_2))

=== SIN RAG ===

En el contexto de sistemas distribuidos y computación en la nube, los patrones de orquestación y memoria se enseñan
como estrategias para gestionar flujos de trabajo y datos de manera eficiente. Los patrones de orquestación (como 
Sagas o Coreografía) coordinan servicios o microservicios para ejecutar procesos complejos, asegurando consistencia
y tolerancia a fallos. Los patrones de memoria (como Caché, Memoización o Almacenamiento en Memoria) optimizan el 
acceso a datos reduciendo latencias y mejorando el rendimiento, mediante técnicas como almacenamiento temporal o 
estructuras de datos en RAM. Ambos son clave para diseñar sistemas escalables y resilientes.

=== CON RAG ===

Según el contexto, en la Clase 3 se abordan patrones de arquitectura con LLMs que incluyen orquestación de 
herramientas y gestión de memoria.

### Qué acabamos de ver

- **Sin RAG**, el LLM responde con lo que "sabe" del curso, es decir, alucina — porque el curso no está en sus datos de entrenamiento.
- **Con RAG**, la respuesta está anclada a los chunks recuperados. Fiel al corpus, o dice "no lo sé".

Esta es la esencia de RAG en una comparación de 6 líneas de código.

## §6 — Bonus: la versión de producción

Lo que hicimos funciona pero **no escala**. Con 10 millones de chunks, calcular coseno contra todos en Python es inviable.

La solución: **extensiones nativas de búsqueda vectorial** en el motor de la base de datos. Con `sqlite-vec` (extensión oficial de SQLite), el retrieval anterior se reduce a:

```sql
SELECT text, distance
FROM chunks_vec
WHERE embedding MATCH ?
ORDER BY distance
LIMIT 3;
```

El motor usa algoritmos optimizados (HNSW, IVF) para hacer la búsqueda aproximada en O(log n) en lugar de O(n).

**Otros vector stores de producción:**

| Stack | Cuándo usarlo |
|---|---|
| **sqlite-vec** | App pequeña/mediana, quieres mantener todo en SQLite |
| **pgvector** | Ya usas Postgres, quieres integridad transaccional con tus datos relacionales |
| **Qdrant / Weaviate** | Self-hosted, alto volumen, filtros complejos con metadata |
| **Pinecone / Voyage** | Managed, no quieres operar infra, pagas por uso |
| **FAISS** | Solo búsqueda en memoria, máximo rendimiento, sin persistencia integrada |

**Mejoras adicionales comunes en producción** (no cubiertas en este demo):

- **Retrieval híbrido**: combinar similitud vectorial + BM25 (búsqueda léxica), fusionando con Reciprocal Rank Fusion.
- **Reranking**: recuperar 20 candidatos y re-ordenarlos con un modelo más caro (cross-encoder) para quedarse con los mejores 3.
- **Query rewriting**: reformular la pregunta del usuario antes de embeber (expansión de siglas, desambiguación).
- **Citaciones**: el LLM devuelve qué chunks usó, para auditoría y verificación.

Todo esto lo implementa `antapaccay-demo`. Lo que hicimos hoy es el **núcleo común** de todos esos sistemas.

In [15]:
conn.close()